# Tree Species Classification Pipeline (DINOv2)
This notebook runs the complete workflow:
1. Crop tree crowns from orthomosaic
2. Run species prediction using a trained DINOv2 classifier
3. Evaluate model performance
4. Export labeled GeoJSON and KML for visualization

## Import libraries

In [ ]:
import os
import torch
import timm
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import simplekml

from shapely.geometry import mapping
from rasterio.mask import mask
from torchvision import transforms
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)


## Configuration

In [ ]:
ORTHO_FOLDER = "data/orthomosaics"
POLYGON_FOLDER = "data/crown_polygons"
MODEL_PATH = "models/best_dinov2_linear.pth"

OUTPUT_DIR = "outputs"
CROP_DIR = os.path.join(OUTPUT_DIR, "cropped_crowns")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224


## Load trained DINOv2 model

In [ ]:
backbone = timm.create_model(
    "vit_base_patch14_dinov2",
    pretrained=False,
    num_classes=0,
    img_size=224
)

classifier = torch.nn.Linear(768, 2)

model = torch.nn.Sequential(backbone, classifier)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

model = model.to(DEVICE)
model.eval()


## Image preprocessing

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])


## Step 1: Crop tree crowns from orthomosaic

In [ ]:
for gj in os.listdir(POLYGON_FOLDER):

    if not gj.endswith(".geojson"):
        continue

    prefix = os.path.splitext(gj)[0]
    gdf = gpd.read_file(os.path.join(POLYGON_FOLDER, gj))

    for ortho in os.listdir(ORTHO_FOLDER):

        if not ortho.endswith(".tif"):
            continue

        ortho_path = os.path.join(ORTHO_FOLDER, ortho)

        with rasterio.open(ortho_path) as src:

            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)

            for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):

                crown_id = row["id"] if "id" in row else idx
                filename = f"{prefix}_{int(crown_id):03d}.tif"

                out_path = os.path.join(CROP_DIR, filename)

                try:
                    out_img, out_transform = mask(
                        src,
                        [mapping(row.geometry)],
                        crop=True
                    )

                    meta = src.meta.copy()
                    meta.update({
                        "height": out_img.shape[1],
                        "width": out_img.shape[2],
                        "transform": out_transform
                    })

                    with rasterio.open(out_path, "w", **meta) as dst:
                        dst.write(out_img)

                except:
                    continue


## Step 2: Predict tree species

In [ ]:
results = []

for img_name in tqdm(os.listdir(CROP_DIR)):

    img_path = os.path.join(CROP_DIR, img_name)

    try:

        with rasterio.open(img_path) as src:
            img = src.read([1,2,3]).transpose(1,2,0)

        img = np.clip(img,0,255).astype(np.uint8)

        input_tensor = transform(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(input_tensor)
            prob = torch.softmax(logits,dim=1)

        pred = torch.argmax(prob,dim=1).item()
        confidence = prob[0][pred].item()

        label = "acacia" if pred == 0 else "non_acacia"

        results.append({
            "image_name": img_name,
            "label": label,
            "confidence": confidence
        })

    except:
        continue

df_pred = pd.DataFrame(results)
df_pred.to_csv(os.path.join(OUTPUT_DIR,"predictions.csv"),index=False)

df_pred.head()


## Step 3: Model evaluation (optional if ground truth exists)

In [ ]:
GROUND_TRUTH_CSV = "data/labels/ground_truth.csv"

if os.path.exists(GROUND_TRUTH_CSV):

    df_true = pd.read_csv(GROUND_TRUTH_CSV)

    df_eval = df_pred.merge(df_true, on="image_name", how="inner")

    label_map = {"acacia":0, "non_acacia":1}

    y_true = df_eval["label_y"].map(label_map)
    y_pred = df_eval["label_x"].map(label_map)
    y_prob = df_eval["confidence"]

    accuracy = (y_true == y_pred).mean()
    print("Accuracy:", accuracy)

    cm = confusion_matrix(y_true, y_pred)

    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=["acacia","non_acacia"],
                yticklabels=["acacia","non_acacia"])

    plt.title("Confusion Matrix")
    plt.show()

    print(classification_report(
        y_true, y_pred,
        target_names=["acacia","non_acacia"]
    ))

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1],'--')
    plt.title(f"ROC Curve (AUC={roc_auc:.3f})")
    plt.show()

else:
    print("Ground truth file not found — skipping evaluation")


## Step 4: Attach predictions to polygons

In [ ]:
all_polys = []

for gj in os.listdir(POLYGON_FOLDER):

    if not gj.endswith(".geojson"):
        continue

    prefix = os.path.splitext(gj)[0]
    g = gpd.read_file(os.path.join(POLYGON_FOLDER, gj))

    g["image_name"] = g["id"].apply(
        lambda x: f"{prefix}_{int(x):03d}.tif"
    )

    all_polys.append(g)

gdf = pd.concat(all_polys, ignore_index=True)

gdf = gdf.merge(df_pred, on="image_name", how="left")

gdf.to_file(os.path.join(OUTPUT_DIR,"species_labeled.geojson"), driver="GeoJSON")


## Step 5: Export KML for Google Earth

In [ ]:
gdf_wgs = gdf.to_crs(epsg=4326)

kml = simplekml.Kml()

for _, row in gdf_wgs.iterrows():

    if pd.isna(row["label"]):
        continue

    geom = row.geometry

    pol = kml.newpolygon(
        name=f"{row['label']} ({row['confidence']:.2f})",
        outerboundaryis=list(geom.exterior.coords)
    )

kml.save(os.path.join(OUTPUT_DIR,"species_map.kml"))

print("KML exported")
